# Fooocus Sunside — Colab

### Кроки
1. **Runtime → GPU** → **Restart session**
2. **Run all** (перший раз **10–20 хв**)
3. Дочекайся в launch-комірці: **`✅ App started successful`**
4. Клікни **`https://….gradio.live`** у виводі комірки

**Preset:** Juggernaut Ragnarok · без цензури · без Pro mode

| Що бачиш у логу | Значення |
|----------------|----------|
| `Downloading: "..."` + **%** | Завантаження моделі з Hugging Face |
| `Installing torch` / `requirements` | Встановлення залежностей (pip показує прогрес) |
| `Running on public URL` | Gradio Share готовий (ще чекай worker) |
| `App started successful` | **Можна відкривати UI** |

⚠️ Не відкривай `*.colab.dev`. Не клікай лінк до `App started successful`.

Якщо після «ЗАПУСК» **немає нових рядків 5+ хв** — **Interrupt** → **Restart session** → відкрий ноутбук заново → Run all.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
import zipfile

# Завжди стабільна робоча директорія (фікс getcwd / deleted folder)
os.chdir("/content")
os.makedirs("/content", exist_ok=True)

REPO = "/content/Fooocus"
REPO_URL = "https://github.com/sunsideaspect/foocus_sunside.git"
ZIP_URL = "https://github.com/sunsideaspect/foocus_sunside/archive/refs/heads/main.zip"

print("=" * 60)
print("КРОК 1/3 — Завантаження коду foocus_sunside")
print("=" * 60)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pygit2==1.15.1"], check=True)


def remove_repo(path: str) -> None:
    if os.path.exists(path):
        subprocess.run(["rm", "-rf", path], check=False)


def repo_ok(path: str) -> bool:
    return os.path.isfile(os.path.join(path, "launch.py"))


def clone_with_git() -> bool:
    os.chdir("/content")
    remove_repo(REPO)
    print("→ git clone (спроба)...")
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO],
        capture_output=True,
        text=True,
        cwd="/content",
    )
    if r.returncode != 0:
        print("  git stderr:", (r.stderr or "").strip())
        return False
    return repo_ok(REPO)


def clone_with_zip() -> bool:
    os.chdir("/content")
    remove_repo(REPO)
    zip_path = "/content/foocus_sunside_main.zip"
    print("→ ZIP fallback (якщо git не працює)...")
    r = subprocess.run(
        ["wget", "--progress=dot:giga", "-O", zip_path, ZIP_URL],
        cwd="/content",
    )
    if r.returncode != 0:
        return False
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/content")
    extracted = "/content/foocus_sunside-main"
    if not os.path.isdir(extracted):
        print("  Папка після ZIP не знайдена:", extracted)
        return False
    shutil.move(extracted, REPO)
    return repo_ok(REPO)


ok = False
for attempt in range(1, 4):
    if clone_with_git():
        ok = True
        print(f"✅ Код завантажено (git, спроба {attempt})")
        break
    print(f"  Повтор git через 8 с ({attempt}/3)...")
    time.sleep(8)

if not ok:
    ok = clone_with_zip()
    if ok:
        print("✅ Код завантажено (ZIP)")

if not ok:
    raise RuntimeError(
        "Не вдалось завантажити foocus_sunside. Runtime → Restart session → Run all"
    )

os.chdir(REPO)
os.environ["TORCH_COMMAND"] = (
    "pip install torch torchvision torchaudio "
    "--index-url https://download.pytorch.org/whl/cu124"
)

print("\n" + "=" * 60)
print("КРОК 2/3 — Перевірка GPU")
print("=" * 60)

import torch

print("Torch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU не знайдено. Runtime → GPU → Restart session → Run all")
print("✅ GPU:", torch.cuda.get_device_name(0))
print("\n→ Далі запусти launch-комірку (КРОК 3).")

In [ ]:
import os
import stat
import subprocess
import sys

import torch

os.chdir("/content")
os.makedirs("/content", exist_ok=True)

REPO = "/content/Fooocus"
if not os.path.isfile(os.path.join(REPO, "launch.py")):
    raise RuntimeError("Спочатку запусти попередню комірку (клон репозиторію).")
os.chdir(REPO)

os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
os.environ["LAUNCH_LIVE_OUTPUT"] = "1"  # pip / install — живий вивід з %

vram_mb = int(torch.cuda.get_device_properties(0).total_memory / (1024 ** 2))
vram_flag = "--always-high-vram" if vram_mb >= 20000 else "--always-normal-vram"

print("=" * 60)
print("КРОК 3/3 — Запуск Fooocus")
print("=" * 60)
print(f"VRAM: {vram_mb} MB → {vram_flag}")
print("\nЩо буде в логу:")
print("  • Installing torch / requirements — pip (з прогресом)")
print("  • Downloading: ... — моделі SDXL (прогрес-бар %)")
print("  • Running on public URL — зʼявиться gradio.live")
print("  • App started successful — UI готовий\n")

print("→ Крок A: фікс numpy (Colab 3.12)...")


def run_live(argv, cwd=REPO, label=""):
    if label:
        print(label, flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["LAUNCH_LIVE_OUTPUT"] = "1"
    p = subprocess.Popen(
        argv,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"Команда завершилась з кодом {p.returncode}: {' '.join(argv)}")


run_live(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "numpy==1.26.4"],
    cwd="/content",
)
print("→ Крок B: gradio + starlette (без import — фікс Colab)...")
run_live(
    [sys.executable, "-m", "pip", "install",
     "starlette>=0.27.0,<1.0.0", "fastapi>=0.100.0,<0.115.0", "gradio==3.41.2"],
    cwd=REPO,
)


def gradio_pkg_dir() -> str:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "show", "gradio"],
        capture_output=True,
        text=True,
        check=True,
    )
    for line in r.stdout.splitlines():
        if line.startswith("Location:"):
            return os.path.join(line.split(":", 1)[1].strip(), "gradio")
    raise RuntimeError("gradio not found after pip install")


print("→ Крок C: frpc для Gradio Share...")
frpc = os.path.join(gradio_pkg_dir(), "frpc_linux_amd64_v0.2")
if not (os.path.isfile(frpc) and os.path.getsize(frpc) > 1_000_000):
    print("→ frpc для Gradio Share...")
    subprocess.run(
        ["wget", "--progress=dot:giga", "-O", frpc,
         "https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_amd64"],
        check=True,
    )
    os.chmod(frpc, stat.S_IRWXU)

launch_args = [
    sys.executable, "-u", "launch.py",
    "--preset", "realistic_juggernaut_ragnarok",
    "--disable-censor",
    "--disable-pro-mode",
    "--disable-preset-selection",
    "--share",
    vram_flag,
    "--disable-in-browser",
]

print("\n" + "=" * 60)
print("ЗАПУСК Fooocus (лог нижче в реальному часі)")
print("Перший раз: torch ~5-10 хв, моделі ~10-15 хв")
print("=" * 60 + "\n")

os.chdir(REPO)
run_live(launch_args, cwd=REPO)
print("\n✅ Fooocus завершив роботу (нормально лише якщо ти сам зупинив сесію).")